# Milestone 2 - Results: final table + CLIP zero-shot baseline

Builds the Milestone 2 table from the `RESULT` lines printed by the six `M2_run_*.ipynb` notebooks (section 7 of each), and runs the CLIP zero-shot baseline. No training here: a T4 is enough (zero-shot takes a few minutes).

- Model selection: **training-domain validation set** (fair), computed in each run notebook with the same rule as DomainBed's `collect_results`.
- **PACS test_env mapping**: `0=art_painting(A)`, `1=cartoon(C)`, `2=photo(P)`, `3=sketch(S)`.

## 1. Paste the results
The output of section 7 of each run notebook.

In [ ]:
# Paste here the output of section 7 of every M2_run_*.ipynb (extra text is ignored).
ALL_RESULTS = """

"""
TRIAL_SEEDS = [0, 1, 2]

import json, re
import numpy as np

def parse_results(text):
    """Extract every `RESULT {...}` line from pasted notebook output (anything else is ignored).
    A run that appears twice keeps its last occurrence."""
    out = {}
    for m in re.finditer(r'RESULT (\{.*\})', text):
        r = json.loads(m.group(1))
        out[(r['method'], r['backbone'], r['seed'], r['te'])] = r
    return out

fmt = lambda v: f'{np.mean(v):.1f} ± {np.std(v):.1f}' if len(v) > 1 else (f'{v[0]:.1f}' if v else '-')

def table(results, seeds, header=True):
    """Markdown table: per-domain accuracy (mean ± std over seeds), average, cost."""
    rows = ['| Backbone | Method | A | C | P | S | Avg | Seeds | Trainable params | Peak VRAM (GB) | s/step | Best step (median / last) |',
            '|---|---|---|---|---|---|---|---|---|---|---|---|'] if header else []
    combos = sorted({(r['backbone'], r['method']) for r in results.values()},
                    key=lambda bm: (['dinov2', 'clip', 'dinov2_cls'].index(bm[0]),
                                    ['lp', 'lora', 'fullft'].index(bm[1])))
    for bname, method in combos:
        res = {(s, te): r for (m, b, s, te), r in results.items() if m == method and b == bname}
        per_dom = [[res[(s, te)]['acc'] for s in seeds if (s, te) in res] for te in range(4)]
        full = [s for s in seeds if all((s, te) in res for te in range(4))]
        avg = [np.mean([res[(s, te)]['acc'] for te in range(4)]) for s in full]
        r = list(res.values())
        rows.append(f'| {bname} | {method} | ' + ' | '.join(fmt(v) for v in per_dom)
                    + f' | **{fmt(avg)}** | {len(full)} | {r[0]["trainable"]:,d}'
                    + f" | {max(x['mem'] for x in r):.1f} | {np.mean([x['step_t'] for x in r]):.3f}"
                    + f" | {int(np.median([x['best_step'] for x in r]))} / {r[0]['last_step']} |")
    return '\n'.join(rows)

RESULTS = parse_results(ALL_RESULTS)
print(f'{len(RESULTS)} runs parsed (72 expected without the optional ablation).')

## 2. Final table (per domain, mean ± std over seeds, + cost)
Also reports trainable parameters, peak GPU memory, seconds per step and the median step of the selected checkpoint. The markdown output can be pasted straight into the README.

In [ ]:
for bname in ['dinov2', 'clip', 'dinov2_cls']:
    for method in ['lp', 'lora', 'fullft']:
        row = [sum((method, bname, s, te) in RESULTS for te in range(4)) for s in TRIAL_SEEDS]
        if bname == 'dinov2_cls' and not any(row):
            continue
        print(f'{method:7s} {bname:11s} ' + '  '.join(f's{s}: {n}/4' for s, n in zip(TRIAL_SEEDS, row)))
print()
print(table(RESULTS, TRIAL_SEEDS))
print('\nAvg and its std use only seeds with all 4 domains finished.')

## 3. CLIP zero-shot baseline (no training at all)
The natural reference point for CLIP, and the starting point of WiSE-FT in Milestone 3. Two prompt sets:
- `plain`: only `"a photo of a {class}."`;
- `ensemble`: the same five templates for every domain, averaged.

Note: zero-shot is evaluated on the WHOLE domain, while the run notebooks use the 80% `in` split of the held-out domain. No test labels are used for anything except computing accuracy.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install open_clip_torch datasets
import os
LOCAL_DATA = '/content/data'
CLIP_HP = {'clip_model': 'ViT-B-16', 'clip_pretrained': 'laion2b_s34b_b88k'}   # same as the run notebooks

In [ ]:
import os, shutil
from datasets import load_dataset

DATA_DIR = os.path.join(LOCAL_DATA, 'PACS')

def n_images(root):
    if not os.path.isdir(root):
        return 0
    return sum(len(files) for _, _, files in os.walk(root))

if n_images(DATA_DIR) == 9991:
    print('PACS already on local disk, skipping download.')
else:
    shutil.rmtree(DATA_DIR, ignore_errors=True)          # start clean
    ds = load_dataset('flwrlabs/pacs', split='train')    # 9991 images
    class_names = ds.features['label'].names             # authoritative class order
    print('classes:', class_names)
    counts = {}
    for i, ex in enumerate(ds):
        domain = ex['domain']                            # art_painting | cartoon | photo | sketch
        cls    = class_names[ex['label']]
        out_dir = os.path.join(DATA_DIR, domain, cls)
        os.makedirs(out_dir, exist_ok=True)
        idx = counts.get((domain, cls), 0); counts[(domain, cls)] = idx + 1
        ex['image'].convert('RGB').save(os.path.join(out_dir, f'{idx:05d}.jpg'), 'JPEG', quality=95)
        if i % 2000 == 0:
            print('written', i, flush=True)

print('PACS ready at', DATA_DIR)
for dom in sorted(os.listdir(DATA_DIR)):
    print(f'{dom:14s} {n_images(os.path.join(DATA_DIR, dom))} img')
assert n_images(DATA_DIR) == 9991

In [ ]:
import gc, open_clip, torch
import numpy as np
from torchvision.datasets import ImageFolder

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_HP['clip_model'], pretrained=CLIP_HP['clip_pretrained'], device='cuda')
model.eval()
tokenizer = open_clip.get_tokenizer(CLIP_HP['clip_model'])

classes = ImageFolder(os.path.join(DATA_DIR, 'photo')).classes   # same order DomainBed uses
PROMPTS = {
    'plain':    ['a photo of a {}.'],
    'ensemble': ['a photo of a {}.', 'a painting of a {}.', 'a cartoon of a {}.',
                 'a sketch of a {}.', 'an image of a {}.'],
}

@torch.no_grad()
def text_head(templates):
    ws = []
    for c in classes:
        t = model.encode_text(tokenizer([tp.format(c) for tp in templates]).cuda())
        t = t / t.norm(dim=-1, keepdim=True)
        t = t.mean(0)
        ws.append(t / t.norm())
    return torch.stack(ws)                                          # (7, 512)

@torch.no_grad()
def zs_acc(domain, W):
    ds = ImageFolder(os.path.join(DATA_DIR, domain), transform=preprocess)
    assert ds.classes == classes
    dl = torch.utils.data.DataLoader(ds, batch_size=256, num_workers=8)
    correct = total = 0
    for xb, yb in dl:
        f = model.encode_image(xb.cuda())
        f = f / f.norm(dim=-1, keepdim=True)
        correct += (f @ W.t()).argmax(1).eq(yb.cuda()).sum().item()
        total += len(yb)
    return 100 * correct / total

zs_rows = ['| CLIP zero-shot | A | C | P | S | Avg |', '|---|---|---|---|---|---|']
heads = {}
for pname, templates in PROMPTS.items():
    W = heads[pname] = text_head(templates)
    accs = [zs_acc(d, W) for d in ['art_painting', 'cartoon', 'photo', 'sketch']]
    zs_rows.append(f'| {pname} | ' + ' | '.join(f'{a:.1f}' for a in accs) + f' | **{np.mean(accs):.1f}** |')
print('\n'.join(zs_rows))

del model; gc.collect(); torch.cuda.empty_cache()

## 4. Results for the README
Paste the tables printed above. Reference points:

| Model | A | C | P | S | Avg |
|---|---|---|---|---|---|
| M1: ERM ResNet-50 (ours) | 82.5 | 80.9 | 97.8 | 78.3 | 84.9 |
| M1: ERM ResNet-50 (paper) | 84.7 | 80.8 | 97.2 | 79.3 | 85.5 |

Questions the write-up should answer:
- Does every adaptation of DINOv2/CLIP beat the M1 ResNet (84.9)? Where is the gain largest? (Expect: sketch and art.)
- Linear probe vs LoRA vs full FT: accuracy against cost (trainable params, VRAM, s/step). Are the differences larger than the std over seeds?
- DINOv2 vs CLIP: is the gap consistent across domains? Does the optional `dinov2_cls` ablation change it?
- How far is fine-tuned CLIP from zero-shot CLIP? If full FT barely beats (or loses to) zero-shot on some domain, that is the motivation for WiSE-FT in Milestone 3.
- If the best step is often the last checkpoint, training was too short: say so, or rerun with more steps.